# Riley sobre GuitarSet

Corre el modelo de guitarra de Xavier Riley (dos checkpoints: `guitar-gaps` y `guitar-fl`) sobre los 144 fragmentos de bossa nova y cantautor de GuitarSet, y devuelve las notas en `riley-guitarset.zip` para medirlas en la Mac con `bench.py`.

Antes de ejecutar: menú Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4. Después, Entorno de ejecución > Ejecutar todas. Al final se descarga el zip solo.

In [ ]:
import torch
assert torch.cuda.is_available(), "No hay GPU: cambia el tipo de entorno de ejecución a T4 y vuelve a ejecutar"
print(torch.cuda.get_device_name(0))

In [ ]:
# Mismas versiones que se probaron en la Mac.
!pip -q install --no-deps "git+https://github.com/xavriley/hf_midi_transcription.git@96f6797881e9497cbfc8f8e5deccea9c1f2f7adc" "git+https://github.com/xavriley/piano_transcription_inference.git@7568dc7f78b625e40cf9776e2806d164006610e3"
!pip -q install torchlibrosa pretty_midi mido resampy

In [ ]:
# Sólo el audio del micrófono: 657 MB.
!wget -q -O audio_mono-mic.zip "https://zenodo.org/api/records/3371780/files/audio_mono-mic.zip/content"
!unzip -q -o audio_mono-mic.zip -d audio && rm audio_mono-mic.zip
import glob
wavs = sorted(w for w in glob.glob("audio/*_mic.wav") if w.split("/")[-1].split("_")[1][:2] in ("BN", "SS"))
print(len(wavs), "fragmentos")

In [ ]:
from huggingface_hub import hf_hub_download
from hf_midi_transcription import MidiTranscriptionModel
import pretty_midi, csv, os, time

checkpoints = {
    "gaps": hf_hub_download("xavriley/midi-transcription-models", "guitar-gaps.pth"),
    "fl": hf_hub_download("xavriley/midi-transcription-models", "guitar-fl.pth"),
}
os.makedirs("midi", exist_ok=True)
for name, path in checkpoints.items():
    model = MidiTranscriptionModel(device="cuda", instrument="guitar", checkpoint_path=path)
    t0 = time.time()
    with open(f"riley-{name}.csv", "w", newline="") as f:
        out = csv.writer(f)
        out.writerow(["file", "start", "end", "pitch", "velocity"])
        for i, wav in enumerate(wavs, 1):
            stem = os.path.basename(wav)[: -len("_mic.wav")]
            mid = f"midi/{name}-{stem}.mid"
            model.transcribe(wav, mid)
            for inst in pretty_midi.PrettyMIDI(mid).instruments:
                for n in inst.notes:
                    out.writerow([stem, round(n.start, 4), round(n.end, 4), n.pitch, n.velocity])
            if i % 24 == 0:
                print(name, i, "/", len(wavs))
    print(name, "listo en", round(time.time() - t0), "s")

In [ ]:
!zip -q riley-guitarset.zip riley-gaps.csv riley-fl.csv
from google.colab import files
files.download("riley-guitarset.zip")